# Incident Response Runbook: Discovery - Network Share Discovery

**Tactic:** Discovery
**Technique:** Network Share Discovery (T1135)
**Severity:** MEDIUM

## Overview

This runbook provides step-by-step incident response procedures for Network Share Discovery activities.

## Incident Response Phases

1. **Detection & Analysis**
2. **Containment**
3. **Eradication**
4. **Recovery**
5. **Post-Incident Activities**


## Phase 1: Detection & Analysis

### Objectives
- Confirm the incident
- Identify the scope
- Assess impact


In [ ]:
import json
import re
from datetime import datetime
import sys
import os

# Add the project root to the Python path
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..', '..')))

from splunk.splunk_data_collector import SplunkDataCollector
from crowdstrike.crowdstrike_response import CrowdStrikeResponse
from iris.iris_integration import IRISIntegration
from misp.misp_integration import MISPIntegration
from shuffle.shuffle_integration import ShuffleIntegration

# Initialize integrations
splunk = SplunkDataCollector()
crowdstrike = CrowdStrikeResponse()
iris = IRISIntegration()
misp = MISPIntegration()
shuffle = ShuffleIntegration()

# Step 1: Detection & Analysis
print("=" * 60)
print("STEP 1: Detection & Analysis")
print("=" * 60)

detection_time = datetime.now().isoformat()

# Query Splunk for network share discovery indicators
print(f"\n[QUERY] Searching Splunk for network share discovery indicators...")
splunk_query = '''
index=* (sourcetype=WinEventLog:Security EventCode=5140 OR EventCode=5145)
OR (sourcetype=linux_secure "mount" OR "smb" OR "cifs")
OR (sourcetype=network "SMB" OR "CIFS" OR "NETLOGON")
| eval share_path=coalesce(ShareName, Share_Path, ObjectName)
| where share_path != null
| stats count by host, user, share_path, _time
| where count > 10
'''

try:
    splunk_results = splunk.search_events(splunk_query, timeframe="-24h")
    print(f"   Found {len(splunk_results)} potential network share discovery events")
except Exception as e:
    print(f"   Splunk query failed: {e}")
    splunk_results = []

# Extract affected systems and suspicious activities
affected_systems = []
suspicious_activities = []
splunk_indicators = []

for event in splunk_results:
    system_info = {
        'hostname': event.get('host', 'unknown'),
        'user': event.get('user', 'unknown'),
        'share_path': event.get('share_path', ''),
        'event_count': event.get('count', 0),
        'last_seen': event.get('_time', detection_time)
    }
    affected_systems.append(system_info)

    # Extract indicators
    if event.get('share_path'):
        splunk_indicators.append({
            'type': 'network_share',
            'value': event.get('share_path'),
            'context': f"Accessed by {event.get('user', 'unknown')} on {event.get('host', 'unknown')}"
        })

# Query CrowdStrike for endpoint detection
print(f"\n[QUERY] Checking CrowdStrike for network share discovery detections...")
try:
    cs_detections = crowdstrike.get_detections(
        filter="technique:'Network Share Discovery'",
        start_time="-24h"
    )
    cs_affected_hosts = []
    for detection in cs_detections:
        host_info = {
            'device_id': detection.get('device_id', ''),
            'hostname': detection.get('hostname', ''),
            'detection_id': detection.get('detection_id', ''),
            'technique': detection.get('technique', ''),
            'severity': detection.get('severity', 0)
        }
        cs_affected_hosts.append(host_info)

        # Merge with Splunk data
        existing_system = next((s for s in affected_systems if s['hostname'] == host_info['hostname']), None)
        if existing_system:
            existing_system.update(host_info)
        else:
            affected_systems.append(host_info)

    print(f"   Found {len(cs_affected_hosts)} CrowdStrike detections")
except Exception as e:
    print(f"   CrowdStrike query failed: {e}")
    cs_affected_hosts = []

# Enrich with threat intelligence from MISP
print(f"\n[ENRICHMENT] Checking MISP for related threat intelligence...")
misp_results = []
try:
    for indicator in splunk_indicators:
        misp_search = misp.search_iocs(indicator['value'])
        if misp_search:
            misp_results.extend(misp_search)
            print(f"   Found threat intelligence for {indicator['value']}")
except Exception as e:
    print(f"   MISP enrichment failed: {e}")

# Create IRIS case
print(f"\n[CASE] Creating IRIS incident case...")
try:
    incident_data = {
        'title': f'Network Share Discovery Incident - {len(affected_systems)} systems',
        'description': f'Detected network share discovery activities on {len(affected_systems)} systems',
        'severity': 'MEDIUM',
        'tactic': 'Discovery',
        'technique': 'Network Share Discovery (T1135)',
        'affected_systems': affected_systems,
        'indicators': splunk_indicators,
        'threat_intel': misp_results,
        'detection_time': detection_time
    }
    incident_id = iris.create_case(incident_data)
    print(f"   Created IRIS case: {incident_id}")
except Exception as e:
    print(f"   IRIS case creation failed: {e}")
    incident_id = f"TEMP-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

print(f"\n Detection complete:")
print(f"  - {len(affected_systems)} affected systems identified")
print(f"  - {len(splunk_indicators)} network share indicators found")
print(f"  - {len(misp_results)} threat intelligence matches")
print(f"  - IRIS case {incident_id} created")


## Phase 2: Containment

### Objectives
- Stop the attack
- Isolate affected systems
- Prevent spread


In [ ]:
# Step 2: Containment
print("\n" + "=" * 60)
print("STEP 2: Containment")
print("=" * 60)

containment_time = datetime.now().isoformat()
isolated_hosts = []
blocked_shares = []
disabled_accounts = []

# Isolate affected systems
print(f"\n[ACTION] Isolating affected systems...")
for system in affected_systems:
    try:
        if 'device_id' in system:
            isolation_result = crowdstrike.isolate_host(system['device_id'])
            if isolation_result.get('status') == 'isolated':
                isolated_hosts.append(system['hostname'])
                print(f"   Isolated host: {system['hostname']}")
    except Exception as e:
        print(f"   Host isolation failed for {system.get('hostname', 'unknown')}: {e}")

# Block suspicious network shares
print(f"\n[ACTION] Blocking suspicious network shares...")
for indicator in splunk_indicators:
    try:
        if indicator['type'] == 'network_share':
            block_result = shuffle.block_network_share(indicator['value'])
            if block_result.get('status') == 'blocked':
                blocked_shares.append(indicator['value'])
                print(f"   Blocked network share: {indicator['value']}")
    except Exception as e:
        print(f"   Network share blocking failed for {indicator['value']}: {e}")

# Disable suspicious user accounts
print(f"\n[ACTION] Disabling suspicious user accounts...")
unique_users = set()
for system in affected_systems:
    if system.get('user') and system['user'] != 'unknown':
        unique_users.add(system['user'])

for user in unique_users:
    try:
        disable_result = shuffle.disable_user_account(user)
        if disable_result.get('status') == 'disabled':
            disabled_accounts.append(user)
            print(f"   Disabled account: {user}")
    except Exception as e:
        print(f"   Account disable failed for {user}: {e}")

# Enable enhanced monitoring
print(f"\n[ACTION] Enabling enhanced monitoring...")
monitoring_rules = []
try:
    # Enable CrowdStrike network share monitoring
    cs_monitoring = crowdstrike.enable_enhanced_monitoring('network_share_discovery')
    if cs_monitoring.get('status') == 'enabled':
        monitoring_rules.append('CrowdStrike network share monitoring')
        print(f"   Enabled CrowdStrike network share monitoring")

    # Enable Splunk correlation rules
    splunk_monitoring = splunk.enable_correlation_rule('network_share_anomaly')
    if splunk_monitoring.get('status') == 'enabled':
        monitoring_rules.append('Splunk network share correlation')
        print(f"   Enabled Splunk network share correlation rules")
except Exception as e:
    print(f"   Enhanced monitoring setup failed: {e}")

# Update IRIS case with containment actions
print(f"\n[ACTION] Updating IRIS case with containment actions...")
try:
    containment_summary = {
        'isolated_hosts': len(isolated_hosts),
        'blocked_shares': len(blocked_shares),
        'disabled_accounts': len(disabled_accounts),
        'monitoring_enabled': len(monitoring_rules),
        'containment_time': containment_time
    }
    iris.update_case(incident_id, 'containment_complete', containment_summary)
    print(f"   Updated IRIS case {incident_id} with containment results")
except Exception as e:
    print(f"   IRIS case update failed: {e}")

print(f"\n Containment complete:")
print(f"  - {len(isolated_hosts)} hosts isolated")
print(f"  - {len(blocked_shares)} network shares blocked")
print(f"  - {len(disabled_accounts)} accounts disabled")
print(f"  - {len(monitoring_rules)} enhanced monitoring rules enabled")


## Phase 3: Eradication

### Objectives
- Remove malicious artifacts
- Close vulnerabilities
- Verify threat removal


In [ ]:
# Step 3: Eradication
print("\n" + "=" * 60)
print("STEP 3: Eradication")
print("=" * 60)

removed_artifacts = []
patched_systems = []
reset_credentials = []

# Remove malicious artifacts
print(f"\n[ACTION] Removing malicious artifacts...")
for system in affected_systems:
    try:
        if 'device_id' in system:
            # Remove any discovered malware or tools
            artifact_removal = crowdstrike.remove_discovery_artifacts(system['device_id'])
            if artifact_removal.get('status') == 'removed':
                removed_artifacts.extend(artifact_removal.get('removed_files', []))
                print(f"   Removed {len(artifact_removal.get('removed_files', []))} artifacts from {system['hostname']}")
    except Exception as e:
        print(f"   Artifact removal failed for {system.get('hostname', 'unknown')}: {e}")

# Reset compromised credentials
print(f"\n[ACTION] Resetting compromised credentials...")
for account in disabled_accounts:
    try:
        reset_result = shuffle.reset_user_password(account)
        if reset_result.get('status') == 'reset':
            reset_credentials.append(account)
            print(f"   Reset password for account: {account}")
    except Exception as e:
        print(f"   Password reset failed for {account}: {e}")

# Patch vulnerabilities
print(f"\n[ACTION] Patching system vulnerabilities...")
for system in affected_systems:
    try:
        if 'device_id' in system:
            patch_result = crowdstrike.apply_security_patches(system['device_id'])
            if patch_result.get('status') == 'patched':
                patched_systems.append(system['hostname'])
                print(f"   Applied security patches to {system['hostname']}")
    except Exception as e:
        print(f"   Patching failed for {system.get('hostname', 'unknown')}: {e}")

# Clean network share access logs
print(f"\n[ACTION] Cleaning network share access logs...")
cleaned_logs = []
for system in affected_systems:
    try:
        if 'device_id' in system:
            log_cleaning = crowdstrike.clean_share_access_logs(system['device_id'])
            if log_cleaning.get('status') == 'cleaned':
                cleaned_logs.append(system['hostname'])
                print(f"   Cleaned share access logs on {system['hostname']}")
    except Exception as e:
        print(f"   Log cleaning failed for {system.get('hostname', 'unknown')}: {e}")

# Verify threat removal
print(f"\n[ACTION] Verifying threat removal...")
verification_results = []
for system in affected_systems:
    try:
        if 'device_id' in system:
            verify_result = crowdstrike.verify_discovery_removal(system['device_id'])
            verification_results.append({
                'system': system['hostname'],
                'status': 'clean' if verify_result.get('discovery_detected', True) == False else 'threats_remaining',
                'remaining_indicators': verify_result.get('remaining_indicators', 0)
            })
            status = "" if verify_result.get('discovery_detected', True) == False else ""
            print(f"  {status} Verification for {system['hostname']}: {verify_result.get('remaining_indicators', 0)} discovery indicators remaining")
    except Exception as e:
        print(f"   Verification failed for {system.get('hostname', 'unknown')}: {e}")

print(f"\n Eradication complete:")
print(f"  - {len(removed_artifacts)} malicious artifacts removed")
print(f"  - {len(reset_credentials)} credentials reset")
print(f"  - {len(patched_systems)} systems patched")
print(f"  - {len(cleaned_logs)} systems had logs cleaned")
print(f"  - {len([v for v in verification_results if v['status'] == 'clean'])} systems verified clean")


## Phase 4: Recovery

### Objectives
- Restore systems
- Validate functionality
- Monitor for issues


In [ ]:
# Step 4: Recovery
print("\n" + "=" * 60)
print("STEP 4: Recovery")
print("=" * 60)

restored_systems = []
validated_functions = []
restored_access = []

# Restore systems from clean backups
print(f"\n[ACTION] Restoring systems from clean backups...")
for system in affected_systems:
    try:
        if 'device_id' in system:
            restore_result = crowdstrike.restore_from_backup(system['device_id'])
            if restore_result.get('status') == 'restored':
                restored_systems.append(system['hostname'])
                print(f"   Restored {system['hostname']} from clean backup")
    except Exception as e:
        print(f"   System restoration failed for {system.get('hostname', 'unknown')}: {e}")

# Restore legitimate network share access
print(f"\n[ACTION] Restoring legitimate network share access...")
for share in blocked_shares:
    try:
        # Check if share is legitimate before restoring
        share_check = shuffle.validate_network_share(share)
        if share_check.get('legitimate', False):
            restore_result = shuffle.restore_network_share(share)
            if restore_result.get('status') == 'restored':
                restored_access.append(share)
                print(f"   Restored legitimate access to: {share}")
        else:
            print(f"   Share {share} flagged as suspicious, access not restored")
    except Exception as e:
        print(f"   Share access restoration failed for {share}: {e}")

# Re-enable user accounts
print(f"\n[ACTION] Re-enabling legitimate user accounts...")
reenabled_accounts = []
for account in disabled_accounts:
    try:
        # Validate account legitimacy
        account_check = shuffle.validate_user_account(account)
        if account_check.get('legitimate', False):
            enable_result = shuffle.enable_user_account(account)
            if enable_result.get('status') == 'enabled':
                reenabled_accounts.append(account)
                print(f"   Re-enabled account: {account}")
        else:
            print(f"   Account {account} flagged as compromised, not re-enabled")
    except Exception as e:
        print(f"   Account re-enabling failed for {account}: {e}")

# Validate system functionality
print(f"\n[ACTION] Validating system functionality...")
functional_tests = []
for system in affected_systems:
    try:
        if 'device_id' in system:
            test_result = crowdstrike.perform_functional_testing(system['device_id'])
            functional_tests.append({
                'system': system['hostname'],
                'tests_passed': test_result.get('tests_passed', 0),
                'total_tests': test_result.get('total_tests', 0)
            })
            pass_rate = test_result.get('tests_passed', 0) / max(test_result.get('total_tests', 1), 1) * 100
            status = "" if pass_rate >= 95 else ""
            print(f"  {status} Functional tests for {system['hostname']}: {test_result.get('tests_passed', 0)}/{test_result.get('total_tests', 0)} passed ({pass_rate:.1f}%)")
    except Exception as e:
        print(f"   Functional testing failed for {system.get('hostname', 'unknown')}: {e}")

# Monitor for residual issues
print(f"\n[ACTION] Establishing monitoring for residual issues...")
monitoring_alerts = []
try:
    # Set up continuous monitoring
    monitoring_setup = splunk.setup_continuous_monitoring('network_share_discovery')
    if monitoring_setup.get('status') == 'enabled':
        monitoring_alerts.append('Splunk continuous monitoring')
        print(f"   Established continuous Splunk monitoring")

    cs_monitoring = crowdstrike.setup_residual_monitoring('network_share')
    if cs_monitoring.get('status') == 'enabled':
        monitoring_alerts.append('CrowdStrike residual monitoring')
        print(f"   Established CrowdStrike residual monitoring")
except Exception as e:
    print(f"   Residual monitoring setup failed: {e}")

print(f"\n Recovery complete:")
print(f"  - {len(restored_systems)} systems restored from backup")
print(f"  - {len(restored_access)} legitimate network shares restored")
print(f"  - {len(reenabled_accounts)} legitimate accounts re-enabled")
print(f"  - {len([f for f in functional_tests if f['tests_passed'] == f['total_tests']])} systems passed all functional tests")
print(f"  - {len(monitoring_alerts)} residual monitoring systems established")


## Phase 5: Post-Incident Activities

### Objectives
- Document findings
- Implement improvements
- Share lessons learned


In [ ]:
# Step 5: Post-Incident
print("\n" + "=" * 60)
print("STEP 5: Post-Incident")
print("=" * 60)

# Update IRIS case with eradication results
print(f"\n[ACTION] Updating IRIS case with eradication results...")
try:
    eradication_summary = {
        'removed_artifacts': len(removed_artifacts),
        'reset_credentials': len(reset_credentials),
        'patched_systems': len(patched_systems),
        'cleaned_logs': len(cleaned_logs),
        'verification_results': verification_results
    }
    iris.update_case(incident_id, 'eradication_complete', eradication_summary)
    print(f"   Updated IRIS case {incident_id} with eradication results")
except Exception as e:
    print(f"   IRIS case update failed: {e}")

# Update IRIS case with recovery results
print(f"\n[ACTION] Updating IRIS case with recovery results...")
try:
    recovery_summary = {
        'restored_systems': len(restored_systems),
        'restored_access': len(restored_access),
        'reenabled_accounts': len(reenabled_accounts),
        'functional_tests': functional_tests,
        'monitoring_alerts': len(monitoring_alerts)
    }
    iris.update_case(incident_id, 'recovery_complete', recovery_summary)
    print(f"   Updated IRIS case {incident_id} with recovery results")
except Exception as e:
    print(f"   IRIS case update failed: {e}")

# Generate incident report
print(f"\n[ACTION] Generating incident report...")
try:
    incident_report = {
        'incident_id': incident_id,
        'technique': 'Discovery: Network Share Discovery',
        'detection_time': detection_time,
        'affected_systems': len(affected_systems),
        'timeline': {
            'detection': detection_time,
            'containment': containment_time,
            'eradication': datetime.now().isoformat(),
            'recovery': datetime.now().isoformat()
        },
        'actions_taken': {
            'detection': f"Found {len(affected_systems)} systems with network share discovery",
            'containment': f"Isolated {len(isolated_hosts)} hosts, blocked {len(blocked_shares)} shares",
            'eradication': f"Removed {len(removed_artifacts)} artifacts, reset {len(reset_credentials)} credentials",
            'recovery': f"Restored {len(restored_systems)} systems, re-enabled {len(reenabled_accounts)} accounts"
        },
        'indicators': splunk_indicators,
        'threat_intel': misp_results,
        'lessons_learned': [
            "Implement stricter network share access monitoring",
            "Regular audit of share permissions and access patterns",
            "Enhanced detection rules for anomalous share enumeration"
        ]
    }
    iris.generate_report(incident_id, incident_report)
    print(f"   Generated comprehensive incident report for case {incident_id}")
except Exception as e:
    print(f"   Report generation failed: {e}")

# Share IOCs with MISP
print(f"\n[ACTION] Sharing IOCs with MISP...")
try:
    misp_iocs = []
    for indicator in splunk_indicators:
        if indicator.get('type') in ['network_share', 'user']:
            misp_iocs.append({
                'type': indicator['type'],
                'value': indicator['value'],
                'tags': ['network-share-discovery', 'incident-' + str(incident_id)]
            })

    if misp_iocs:
        misp.share_iocs(misp_iocs, f"Network Share Discovery Incident {incident_id}")
        print(f"   Shared {len(misp_iocs)} IOCs with MISP community")
    else:
        print(f"   No new IOCs to share with MISP")
except Exception as e:
    print(f"   MISP IOC sharing failed: {e}")

# Close IRIS case
print(f"\n[ACTION] Closing IRIS case...")
try:
    iris.close_case(incident_id, 'resolved')
    print(f"   Closed IRIS case {incident_id} as resolved")
except Exception as e:
    print(f"   IRIS case closure failed: {e}")

# Final status summary
print(f"\n Post-incident activities complete:")
print(f"  - IRIS case {incident_id} updated and closed")
print(f"  - Incident report generated")
print(f"  - {len(misp_iocs) if 'misp_iocs' in locals() else 0} IOCs shared with threat intelligence community")
print(f"  - All 5 IR phases completed successfully")

# Calculate incident metrics
incident_duration = (datetime.now() - datetime.fromisoformat(detection_time)).total_seconds() / 3600
print(f"\n📊 Incident Metrics:")
print(f"  - Duration: {incident_duration:.2f} hours")
print(f"  - Systems affected: {len(affected_systems)}")
print(f"  - Containment time: {'< 1 hour' if incident_duration < 1 else f'{incident_duration:.1f} hours'}")
print(f"  - Eradication success: {len([v for v in verification_results if v['status'] == 'clean'])}/{len(affected_systems)} systems clean")
print(f"  - Recovery success: {len([f for f in functional_tests if f['tests_passed'] == f['total_tests']])}/{len(affected_systems)} systems fully functional")


## Summary

This incident response runbook has guided you through the complete lifecycle of responding to this incident.

### Key Takeaways
- Rapid detection and response are critical
- Containment prevents further damage
- Thorough eradication prevents reinfection
- Continuous monitoring ensures recovery

### Next Steps
- Review and approve the incident report
- Implement recommended preventive measures
- Schedule follow-up review
- Share lessons learned with the team
